# PathSnap – Exploratory Data Analysis & Route Reconstruction Demo

This notebook demonstrates:
1. Loading and parsing trajectory and road network files.
2. Cleaning and Kalman filtering noisy GPS observations.
3. Running candidate road generation with spatial indexing.
4. HMM and Viterbi sequence route reconstruction.
5. Evaluating route accuracy against ground truth.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from backend.src.data.parser import load_trajectory_bundle
from backend.src.network.graph import RoadNetwork
from backend.src.network.spatial_index import SpatialGridIndex
from backend.src.ml.models import RandomForestModel
from backend.src.hmm.pipeline import RouteReconstructionPipeline

traj_dir = os.path.join('..', 'data', 'raw', 'train', '00000001')
bundle = load_trajectory_bundle(traj_dir, '00000001')
print(f"Loaded Trajectory 00000001:")
print(f" - GPS points: {len(bundle['track'])}")
print(f" - Road nodes: {len(bundle['nodes'])}")
print(f" - Road arcs:  {len(bundle['arcs'])}")
print(f" - Ground truth route arcs: {len(bundle['route'])}")

In [ ]:
# Initialize Road Network & Spatial Grid Index
network = RoadNetwork(bundle['nodes'], bundle['arcs'])
spatial_index = SpatialGridIndex(network)

# Load trained Random Forest Model
rf = RandomForestModel()
model_path = os.path.join('..', 'models', 'saved', 'random_forest.joblib')
rf.load(model_path)

# Run Reconstruction Pipeline
pipeline = RouteReconstructionPipeline(network, spatial_index, ml_model=rf)
result = pipeline.reconstruct(bundle['track'][:500], bundle['route'])

print("Reconstruction Status:", result['status'])
print("Processing Time:", result['processing_time_ms'], "ms")
print("Reconstructed Distance:", result['reconstructed_distance_km'], "km")
print("Continuous Arcs Count:", len(result['continuous_arcs']))
print("Gaps Bridged via Shortest Path:", result['num_gaps_bridged'])
print("Evaluation Metrics:", result['evaluation'])